# GraviProb - Bayesian inference

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviProb/bin/Release/net10.0/Gravicode.Science.GraviProb.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviFrame;
using Gravicode.Science.GraviNum;
using Gravicode.Science.GraviProb;
using Gravicode.Science.GraviProb.Models;

var flips = DataFrame.ReadCsv("../datasets/bayesian_coin.csv");
var trials = flips.RowCount;
var heads = (int)flips.Numeric("outcome").Sum();
Console.WriteLine($"{heads} heads in {trials} flips ({(double)heads / trials:P2})");

## The model

A uniform Beta(1,1) prior with a binomial likelihood. Because Beta is conjugate to the binomial, the exact posterior is available - which lets the sampler be checked against ground truth.

In [ ]:
var model = new BayesianModel()
    .AddDistribution("theta", Distribution.Beta(1, 1))
    .AddObservation("data", DistributionSpec.Binomial(trials, "theta"), heads);

var exact = Distribution.Beta(1, 1).PosteriorAfter(heads, trials - heads);
Console.WriteLine($"exact posterior: {exact.Name}, mean {exact.Mean:F6}, sd {exact.StandardDeviation:F6}");

## Sampling

In [ ]:
var posterior = model.SampleMCMC(iterations: 20_000, chains: 4, warmup: 10_000, seed: 42);
Console.Write(posterior.Summary());

var (low, high) = posterior.HighestDensityInterval("theta", 0.95);
Console.WriteLine($"95% HDI: [{low:F4}, {high:F4}]");
Console.WriteLine($"P(theta > 0.5) = {posterior["theta"].ToArray().Count(v => v > 0.5) / (double)posterior.TotalDraws:P2}");

## Posterior plot

In [ ]:
var draws = posterior["theta"];
var (edges, counts) = Statistics.Histogram(draws, bins: 60);
var width = edges[1] - edges[0];
var centres = new double[counts.Length];
var density = new double[counts.Length];
for (var i = 0; i < counts.Length; i++)
{
    centres[i] = (edges[i] + edges[i + 1]) / 2;
    density[i] = counts[i] / (draws.Size * width);
}

var curveX = NdArray.Linspace(0.01, 0.99, 400).ToArray();
var curveY = curveX.Select(exact.Density).ToArray();

var plot = new ScottPlot.Plot();
var bars = plot.Add.Bars(centres, density);
bars.LegendText = "MCMC draws";
foreach (var bar in bars.Bars) bar.Size = width * 0.9;

var curve = plot.Add.Scatter(curveX, curveY);
curve.LegendText = "exact Beta posterior"; curve.MarkerSize = 0; curve.LineWidth = 3;

var span = plot.Add.HorizontalSpan(low, high);
span.LegendText = "95% HDI";
span.FillColor = ScottPlot.Colors.Orange.WithAlpha(0.15);

plot.Title($"Posterior for theta: {heads} heads in {trials} flips");
plot.XLabel("theta"); plot.YLabel("density");
plot.ShowLegend();
plot.GetPngHtml(900, 550)

## Posterior predictive check

Can the fitted model reproduce the data it was fitted to?

In [ ]:
var replicated = posterior.PosteriorPredictive((v, rng) => rng.Binomial(trials, v["theta"]), draws: 5000, seed: 42);
Console.WriteLine($"replicated: mean {Statistics.Mean(replicated):F2}, sd {Statistics.Std(replicated):F2}");
Console.WriteLine($"observed {heads} sits at percentile {replicated.ToArray().Count(v => v < heads) / 5000.0:P1}");

## A Bayesian network

Explaining away: once the sprinkler accounts for the wet grass, rain becomes less necessary.

In [ ]:
var network = new BayesianNetwork()
    .AddVariable("rain", 0.8, 0.2)
    .AddVariable("sprinkler", 2, new[] { "rain" }, new[] { new[] { 0.6, 0.4 }, new[] { 0.99, 0.01 } })
    .AddVariable("wet", 2, new[] { "rain", "sprinkler" }, new[]
    {
        new[] { 1.00, 0.00 },
        new[] { 0.10, 0.90 },
        new[] { 0.20, 0.80 },
        new[] { 0.01, 0.99 },
    });

Console.WriteLine($"P(rain)                  = {network.Infer("rain")[1]:F4}");
Console.WriteLine($"P(rain | wet)            = {network.Infer("rain", new Dictionary<string, int> { ["wet"] = 1 })[1]:F4}");
Console.WriteLine($"P(rain | wet, sprinkler) = {network.Infer("rain", new Dictionary<string, int> { ["wet"] = 1, ["sprinkler"] = 1 })[1]:F4}");

## Multivariate distributions

A `MultivariateNormal` runs everything off one Cholesky factor: the density, the log determinant and
the sampling. Inverting the covariance directly would be slower and markedly less accurate for an
ill-conditioned matrix — which is exactly when it matters.

Conditioning a normal on part of itself gives another normal, in closed form. That is the entire
mechanism behind Gaussian process regression.


In [ ]:
var mvnMean = NdArray.FromValues([1.0, -2.0]);
var mvnCov = NdArray.FromArray(new double[,] { { 4.0, 1.5 }, { 1.5, 2.0 } });
var mvn = new MultivariateNormal(mvnMean, mvnCov);

var conditioned = mvn.Conditional([0], [1], NdArray.FromValues([1.0]));
Console.WriteLine($"unconditional : mean {mvnMean.At(0):F4}, variance {mvnCov[0, 0]:F4}");
Console.WriteLine($"given x1 = 1  : mean {conditioned.Mean.At(0):F4}, variance {conditioned.Covariance[0, 0]:F4}");
Console.WriteLine("conditioning always reduces the variance - that is what learning something means\n");

var dirichlet = new Dirichlet(2, 3, 5);
Console.WriteLine($"Dirichlet(2,3,5) mean : [{string.Join(", ", dirichlet.Mean.ToArray().Select(v => v.ToString("F3")))}]");
Console.WriteLine($"posterior after counts [10, 5, 0] : [{string.Join(", ", dirichlet.Posterior([10.0, 5.0, 0.0]).Alpha)}]");
Console.WriteLine("conjugate, so the update is addition - which is why it is the default prior");
Console.WriteLine("for anything proportion-shaped\n");

Console.WriteLine("Off-diagonal covariance is NEGATIVE and must be - the components sum to one:");
for (var i = 0; i < 3; i++)
    Console.WriteLine($"  [{string.Join(", ", Enumerable.Range(0, 3).Select(j => dirichlet.Covariance[i, j].ToString("F5")))}]");


## Gaussian processes

A prior on the **function** rather than on the parameters of one. No optimisation is involved: the
posterior is a closed-form Gaussian conditional, and the uncertainty arrives with the prediction.

The band collapses onto the observations and fans out beyond them. The dotted draws are coherent
*functions* — a marginal band says nothing about shape.


In [ ]:
const int observedCount = 12;
var gpX = NdArray.Zeros(observedCount, 1);
var gpY = NdArray.Zeros(observedCount);
for (var i = 0; i < observedCount; i++)
{
    var value = i * 2 * Math.PI / observedCount;
    gpX[i, 0] = value;
    gpY.SetAt(i, Math.Sin(value));
}

var gp = new GaussianProcess(new RbfKernel(lengthScale: 1.0), noise: 1e-6).Fit(gpX, gpY);
Console.WriteLine($"log marginal likelihood = {gp.LogMarginalLikelihood():F4}");

const int gridSize = 200;
var gridX = NdArray.Zeros(gridSize, 1);
var gridValues = new double[gridSize];
for (var i = 0; i < gridSize; i++)
{
    gridValues[i] = -1 + i * 9.0 / (gridSize - 1);
    gridX[i, 0] = gridValues[i];
}

var band = gp.Predict(gridX);
var (lower, upper) = band.Interval(0.95);

var gpPlot = new ScottPlot.Plot();
var fill = gpPlot.Add.FillY(gridValues, lower.ToArray(), upper.ToArray());
fill.LegendText = "95% credible interval";
fill.FillColor = ScottPlot.Colors.SteelBlue.WithAlpha(0.25);

var meanLine = gpPlot.Add.Scatter(gridValues, band.Mean.ToArray());
meanLine.LegendText = "posterior mean";
meanLine.MarkerSize = 0;

var gpDraws = gp.SamplePosterior(gridX, count: 3, new GraviRandom(5));
for (var s = 0; s < 3; s++)
{
    var draw = new double[gridSize];
    for (var i = 0; i < gridSize; i++) draw[i] = gpDraws[s, i];

    var line = gpPlot.Add.Scatter(gridValues, draw);
    line.MarkerSize = 0;
    line.LineWidth = 1;
    line.LinePattern = ScottPlot.LinePattern.Dotted;
    if (s == 0) line.LegendText = "posterior draws";
}

var marks = gpPlot.Add.Scatter(
    Enumerable.Range(0, observedCount).Select(i => gpX[i, 0]).ToArray(), gpY.ToArray());
marks.LineWidth = 0;
marks.MarkerSize = 9;
marks.LegendText = "observations";

gpPlot.Title("Gaussian process posterior");
gpPlot.ShowLegend();
gpPlot.GetPngHtml(850, 500)


## Kalman filter and smoother

Within the linear-Gaussian assumptions the Kalman filter is not a good method, it is *the* method:
the exact posterior over the hidden state, and the minimum-variance estimator among **all**
estimators, not merely linear ones.

Filtering uses only the past, which is what a real-time system can do. Smoothing uses the whole
series and is strictly better — using smoothed states to evaluate a forecasting rule is a
look-ahead error, and a common one.


In [ ]:
var kalman = KalmanFilter.LocalLevel(processVariance: 0.05, observationVariance: 1.0);
var (hiddenLevel, noisy) = kalman.Simulate(200, new GraviRandom(31));

var filtered = kalman.Filter(noisy);
var smoothed = kalman.Smooth(noisy);

double Rmse(IReadOnlyList<double> estimate)
{
    var total = 0.0;
    for (var t = 0; t < estimate.Count; t++)
    {
        var error = estimate[t] - hiddenLevel[t, 0];
        total += error * error;
    }
    return Math.Sqrt(total / estimate.Count);
}

var rawError = 0.0;
for (var t = 0; t < 200; t++)
{
    var error = noisy[t, 0] - hiddenLevel[t, 0];
    rawError += error * error;
}

Console.WriteLine($"raw observations RMSE : {Math.Sqrt(rawError / 200):F4}");
Console.WriteLine($"filtered         RMSE : {Rmse(filtered.Filtered.Select(s => s.Mean.At(0)).ToList()):F4}");
Console.WriteLine($"smoothed         RMSE : {Rmse(smoothed.Select(s => s.Mean.At(0)).ToList()):F4}");
Console.WriteLine($"log likelihood        : {filtered.LogLikelihood:F2}   <- what parameter fitting maximises");

var times = Enumerable.Range(0, 200).Select(t => (double)t).ToArray();
var kalmanPlot = new ScottPlot.Plot();

var obs = kalmanPlot.Add.Scatter(times, Enumerable.Range(0, 200).Select(t => noisy[t, 0]).ToArray());
obs.LineWidth = 0;
obs.MarkerSize = 3;
obs.Color = ScottPlot.Colors.Gray.WithAlpha(0.5);
obs.LegendText = "observations";

var truth = kalmanPlot.Add.Scatter(times, Enumerable.Range(0, 200).Select(t => hiddenLevel[t, 0]).ToArray());
truth.MarkerSize = 0;
truth.LineWidth = 2;
truth.LegendText = "hidden state (never observed)";

var smooth = kalmanPlot.Add.Scatter(times, smoothed.Select(s => s.Mean.At(0)).ToArray());
smooth.MarkerSize = 0;
smooth.LineWidth = 2;
smooth.LegendText = "smoothed estimate";

kalmanPlot.Title("Kalman smoother recovering a hidden state");
kalmanPlot.XLabel("time");
kalmanPlot.ShowLegend();
kalmanPlot.GetPngHtml(850, 450)


## WAIC and LOO

The in-sample answer to "how well does this model predict" is systematically optimistic — a more
flexible model always fits the data it was fitted on better.

LOO comes with a **diagnostic** that WAIC has no equivalent for: the Pareto `k` says whether the
importance reweighting can be trusted. Watch it fire on the badly misspecified model below.


In [ ]:
var comparisonRng = new GraviRandom(41);
var comparisonData = NdArray.Zeros(60);
for (var i = 0; i < 60; i++) comparisonData.SetAt(i, comparisonRng.Normal());

var dataMean = 0.0;
for (var i = 0; i < 60; i++) dataMean += comparisonData.At(i);
dataMean /= 60;

var drawRng = new GraviRandom(43);
var drawnMeans = new double[1500];
for (var s = 0; s < 1500; s++) drawnMeans[s] = dataMean + drawRng.Normal() / Math.Sqrt(60);

NdArray Pointwise(double sigma)
{
    var matrix = NdArray.Zeros(drawnMeans.Length, comparisonData.Size);
    for (var s = 0; s < drawnMeans.Length; s++)
    {
        var normal = new Normal(drawnMeans[s], sigma);
        for (var i = 0; i < comparisonData.Size; i++) matrix[s, i] = normal.LogDensity(comparisonData.At(i));
    }
    return matrix;
}

var candidates = new Dictionary<string, InformationCriterion>();
foreach (var (label, sigma) in new[] { ("sigma = 0.5", 0.5), ("sigma = 1.0", 1.0), ("sigma = 4.0", 4.0) })
{
    var matrix = Pointwise(sigma);
    var waic = ModelComparison.Waic(matrix);
    var loo = ModelComparison.Loo(matrix);
    candidates[label] = waic;

    Console.WriteLine($"{label,-12} WAIC {waic.Estimate,8:F2}  LOO {loo.Criterion.Estimate,8:F2}  " +
                      $"p_eff {waic.EffectiveParameters,5:F2}  reliable {loo.IsReliable}");
}

Console.WriteLine("\nRanked, with the standard error of each DIFFERENCE from the best:");
foreach (var (name, estimate, difference, error) in ModelComparison.Compare(candidates))
    Console.WriteLine($"  {name,-12} {estimate,8:F2}  {(difference == 0 ? "best" : $"+{difference:F2} +/- {error:F2}")}");

Console.WriteLine("\nData came from N(0, 1), and sigma = 1.0 wins. The misspecified sigma = 0.5 reports");
Console.WriteLine("'reliable False' - a few observations dominate its importance weights, so its LOO");
Console.WriteLine("estimate should not be believed. WAIC fails silently in exactly that case.");
